In [10]:
# ============================================================
# AG NEWS CLASSIFICATION USING TRANSFORMER (DISTILBERT)
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

# Run this in terminal before running the program:
#
# pip install torch transformers datasets scikit-learn pandas matplotlib
#===========================================================
# 2. IMPORT LIBRARIES
# ============================================================
!pip install datasets
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# ============================================================
# 3. DEVICE
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)


# ============================================================
# 4. LOAD AG NEWS DATASET
# ============================================================

print("\nLoading AG News Dataset...")

dataset = load_dataset("ag_news")

train_dataset = dataset["train"]

test_dataset = dataset["test"]


print("\nTraining Dataset Size:")

print(len(train_dataset))


print("\nTesting Dataset Size:")

print(len(test_dataset))


# ============================================================
# 5. DEFINE THE FOUR NEWS CATEGORIES
# ============================================================

categories = {

    0: "World",

    1: "Sports",

    2: "Business",

    3: "Technology"

}


print("\nFour News Categories:")

for label, category in categories.items():

    print(label, "-", category)


# ============================================================
# 6. EXPLORE DATASET
# ============================================================

print("\nSample News Articles:\n")


for i in range(5):

    print("Article:", train_dataset[i]["text"])

    print(
        "Category:",
        categories[train_dataset[i]["label"]]
    )

    print("-" * 70)


# ============================================================
# 7. CATEGORY DISTRIBUTION
# ============================================================

labels = train_dataset["label"]


print("\nCategory Distribution:\n")


for label in range(4):

    count = labels.count(label)

    print(

        categories[label],

        ":",

        count

    )


# ============================================================
# 8. LOAD TRANSFORMER TOKENIZER
# ============================================================

MODEL_NAME = "distilbert-base-uncased"


print("\nLoading Transformer Tokenizer...")


tokenizer = AutoTokenizer.from_pretrained(

    MODEL_NAME

)


# ============================================================
# 9. TOKENIZE NEWS ARTICLES
# ============================================================

def tokenize_function(examples):

    return tokenizer(

        examples["text"],

        padding="max_length",

        truncation=True,

        max_length=128

    )


print("\nTokenizing Training Dataset...")


tokenized_train = train_dataset.map(

    tokenize_function,

    batched=True

)


print("\nTokenizing Testing Dataset...")


tokenized_test = test_dataset.map(

    tokenize_function,

    batched=True

)


# ============================================================
# 10. DISPLAY INPUT IDS AND ATTENTION MASKS
# ============================================================

print("\nExample News Article:\n")

print(train_dataset[0]["text"])


print("\nInput IDs:\n")

print(

    tokenized_train[0]["input_ids"]

)


print("\nAttention Mask:\n")

print(

    tokenized_train[0]["attention_mask"]

)


# ============================================================
# 11. REMOVE TEXT COLUMN
# ============================================================

tokenized_train = tokenized_train.remove_columns(

    ["text"]

)


tokenized_test = tokenized_test.remove_columns(

    ["text"]

)


# ============================================================
# 12. RENAME LABEL COLUMN
# ============================================================

tokenized_train = tokenized_train.rename_column(

    "label",

    "labels"

)


tokenized_test = tokenized_test.rename_column(

    "label",

    "labels"

)


# ============================================================
# 13. SET DATA FORMAT
# ============================================================

tokenized_train.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "labels"

    ]

)


tokenized_test.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "labels"

    ]

)


# ============================================================
# 14. CREATE TRAIN AND VALIDATION DATA
# ============================================================

print("\nCreating Training and Validation Datasets...")


split_dataset = tokenized_train.train_test_split(

    test_size=0.1,

    seed=42

)


train_data = split_dataset["train"]

validation_data = split_dataset["test"]


print("\nTraining Samples:")

print(len(train_data))


print("\nValidation Samples:")

print(len(validation_data))


print("\nTesting Samples:")

print(len(tokenized_test))


# ============================================================
# 15. LOAD PRETRAINED TRANSFORMER MODEL
# ============================================================

print("\nLoading Pretrained DistilBERT Model...")


model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=4

)


# ============================================================
# 16. EVALUATION METRICS
# ============================================================

def compute_metrics(prediction):

    logits = prediction.predictions

    labels = prediction.label_ids


    predicted_labels = logits.argmax(

        axis=1

    )


    accuracy = accuracy_score(

        labels,

        predicted_labels

    )


    precision, recall, f1, _ = (

        precision_recall_fscore_support(

            labels,

            predicted_labels,

            average="weighted",

            zero_division=0

        )

    )


    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1

    }


# ============================================================
# 17. TRAINING CONFIGURATION
# ============================================================

training_arguments = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=100,

    load_best_model_at_end=True

)


# ============================================================
# 18. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_arguments,

    train_dataset=train_data,

    eval_dataset=validation_data,

    compute_metrics=compute_metrics

)


# ============================================================
# 19. TRAIN THE MODEL
# ============================================================

print("\n" + "=" * 60)

print("TRAINING STARTED")

print("=" * 60)


trainer.train()


print("\n" + "=" * 60)

print("TRAINING COMPLETED")

print("=" * 60)


# ============================================================
# 20. VALIDATE THE MODEL
# ============================================================

print("\nVALIDATION RESULTS:\n")


validation_results = trainer.evaluate()


for key, value in validation_results.items():

    print(

        key,

        ":",

        value

    )


# ============================================================
# 21. SAVE THE MODEL
# ============================================================

MODEL_FOLDER = "ag_news_model"


trainer.save_model(

    MODEL_FOLDER

)


tokenizer.save_pretrained(

    MODEL_FOLDER

)


print(

    "\nModel saved successfully!"

)


# ============================================================
# 22. TEST DATASET EVALUATION
# ============================================================

print("\n" + "=" * 60)

print("TEST DATASET EVALUATION")

print("=" * 60)


prediction_output = trainer.predict(

    tokenized_test

)


logits = prediction_output.predictions


true_labels = prediction_output.label_ids


predicted_labels = logits.argmax(

    axis=1

)


# ============================================================
# 23. ACCURACY
# ============================================================

accuracy = accuracy_score(

    true_labels,

    predicted_labels

)


print("\nAccuracy:")

print(

    round(

        accuracy * 100,

        2

    ),

    "%"

)


# ============================================================
# 24. PRECISION, RECALL AND F1 SCORE
# ============================================================

precision, recall, f1, _ = (

    precision_recall_fscore_support(

        true_labels,

        predicted_labels,

        average="weighted",

        zero_division=0

    )

)


print("\nPrecision:")

print(

    round(

        precision * 100,

        2

    ),

    "%"

)


print("\nRecall:")

print(

    round(

        recall * 100,

        2

    ),

    "%"

)


print("\nF1 Score:")

print(

    round(

        f1 * 100,

        2

    ),

    "%"

)


# ============================================================
# 25. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 60)

print("CLASSIFICATION REPORT")

print("=" * 60)


print(

    classification_report(

        true_labels,

        predicted_labels,

        target_names=[

            "World",

            "Sports",

            "Business",

            "Technology"

        ],

        zero_division=0

    )

)


# ============================================================
# 26. CONFUSION MATRIX
# ============================================================

print("\n" + "=" * 60)

print("CONFUSION MATRIX")

print("=" * 60)


matrix = confusion_matrix(

    true_labels,

    predicted_labels

)


print("\n")

print(matrix)


# ============================================================
# 27. DISPLAY CONFUSION MATRIX
# ============================================================

display = ConfusionMatrixDisplay(

    confusion_matrix=matrix,

    display_labels=[

        "World",

        "Sports",

        "Business",

        "Technology"

    ]

)


plt.figure(

    figsize=(8, 6)

)


display.plot(

    cmap="Blues",

    values_format="d"

)


plt.title(

    "AG News Classification - Confusion Matrix"

)


plt.show()


# ============================================================
# 28. LOAD SAVED MODEL
# ============================================================

print("\nLoading Saved Model...")


loaded_model = (

    AutoModelForSequenceClassification.from_pretrained(

        MODEL_FOLDER

    )

)


loaded_tokenizer = (

    AutoTokenizer.from_pretrained(

        MODEL_FOLDER

    )

)


loaded_model.to(

    DEVICE

)


loaded_model.eval()


# ============================================================
# 29. PREDICT NEW NEWS ARTICLE
# ============================================================

def predict_news(text):


    inputs = loaded_tokenizer(

        text,

        return_tensors="pt",

        truncation=True,

        padding=True,

        max_length=128

    )


    inputs = {

        key: value.to(DEVICE)

        for key, value in inputs.items()

    }


    with torch.no_grad():


        outputs = loaded_model(

            **inputs

        )


    probabilities = torch.softmax(

        outputs.logits,

        dim=1

    )


    confidence, predicted_label = (

        torch.max(

            probabilities,

            dim=1

        )

    )


    predicted_label = (

        predicted_label.item()

    )


    confidence = (

        confidence.item() * 100

    )


    return (

        categories[predicted_label],

        confidence

    )


# ============================================================
# 30. EXAMPLE PREDICTION
# ============================================================

example_news = (

    "Apple announces a new smartphone "

    "with advanced AI features."

)


category, confidence = (

    predict_news(

        example_news

    )

)


print("\n" + "=" * 60)

print("EXAMPLE PREDICTION")

print("=" * 60)


print(

    "\nInput:",

    example_news

)


print(

    "\nPredicted Category:",

    category

)


print(

    "\nConfidence:",

    round(confidence, 2),

    "%"

)


# ============================================================
# 31. USER INPUT PREDICTION
# ============================================================

print("\n" + "=" * 60)

print("AG NEWS CATEGORY PREDICTION")

print("=" * 60)


print("\nAvailable Categories:")

print("1. World")

print("2. Sports")

print("3. Business")

print("4. Technology")


while True:


    user_news = input(

        "\nEnter a news headline/article "
        "(or type 'exit'): "

    )


    if user_news.lower() == "exit":


        print(

            "\nProgram Ended!"

        )


        break


    predicted_category, confidence = (

        predict_news(

            user_news

        )

    )


    print("\nInput:")

    print(

        user_news

    )


    print("\nPredicted Category:")

    print(

        predicted_category

    )


    print("\nConfidence:")

    print(

        round(confidence, 2),

        "%"

    )

Device: cpu

Loading AG News Dataset...


HfUriError: Invalid HF URI 'hf://datasets/ag_news@eb185aade064a813bc0b7f42de02595523103ca4/.huggingface.yaml'. Repository id must be 'namespace/name', got 'ag_news'.

In [6]:
import pandas as pd

train_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"

test_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"

train_df = pd.read_csv(
    train_url,
    header=None,
    names=["label", "title", "description"]
)

test_df = pd.read_csv(
    test_url,
    header=None,
    names=["label", "title", "description"]
)

print("Training Data Shape:", train_df.shape)
print("Testing Data Shape:", test_df.shape)

train_df.head()

Training Data Shape: (120000, 3)
Testing Data Shape: (7600, 3)


,label,title,description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [7]:
categories = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Technology"
}

print("AG News Categories:\n")

for label, category in categories.items():
    print(label, "-", category)

print("\nCategory Distribution:")

print(train_df["label"].value_counts().sort_index())

AG News Categories:

1 - World
2 - Sports
3 - Business
4 - Technology

Category Distribution:
label
1    30000
2    30000
3    30000
4    30000
Name: count, dtype: int64


In [8]:
train_df["label"] = train_df["label"] - 1

test_df["label"] = test_df["label"] - 1

categories = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Technology"
}

print("Labels converted successfully!")

print(train_df["label"].value_counts().sort_index())

Labels converted successfully!
label
0    30000
1    30000
2    30000
3    30000
Name: count, dtype: int64


In [9]:
train_df["text"] = (
    train_df["title"].astype(str) + " " +
    train_df["description"].astype(str)
)

test_df["text"] = (
    test_df["title"].astype(str) + " " +
    test_df["description"].astype(str)
)

train_df = train_df[["text", "label"]]

test_df = test_df[["text", "label"]]

print(train_df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [11]:
print("News Article:")

print(train_df.iloc[0]["text"])

print("\nLabel:")

print(train_df.iloc[0]["label"])

print("\nCategory:")

print(categories[train_df.iloc[0]["label"]])

News Article:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Label:
2

Category:
Business


In [12]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

print("Loading DistilBERT Tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

Loading DistilBERT Tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\subbu\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\subbu\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [13]:
sample_text = train_df.iloc[0]["text"]

tokens = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=128
)

print("Original Text:\n")
print(sample_text)

print("\nInput IDs:\n")
print(tokens["input_ids"])

print("\nAttention Mask:\n")
print(tokens["attention_mask"])

Original Text:

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Input IDs:

[101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813, 2395, 1005, 1055, 1040, 11101, 2989, 1032, 2316, 1997, 11087, 1011, 22330, 8713, 2015, 1010, 2024, 3773, 2665, 2153, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Attention Mask:

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [15]:
from torch.utils.data import Dataset

class AGNewsDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length):

        self.texts = dataframe["text"].tolist()

        self.labels = dataframe["label"].tolist()

        self.tokenizer = tokenizer

        self.max_length = max_length


    def __len__(self):

        return len(self.texts)


    def __getitem__(self, index):

        text = self.texts[index]

        label = self.labels[index]


        encoding = self.tokenizer(

            text,

            truncation=True,

            padding="max_length",

            max_length=self.max_length,

            return_tensors="pt"

        )


        return {

            "input_ids":
            encoding["input_ids"].flatten(),

            "attention_mask":
            encoding["attention_mask"].flatten(),

            "labels":
            torch.tensor(label, dtype=torch.long)

        }

In [16]:
MAX_LENGTH = 128

train_dataset = AGNewsDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)

test_dataset = AGNewsDataset(
    test_df,
    tokenizer,
    MAX_LENGTH
)

print("Training Samples:", len(train_dataset))

print("Testing Samples:", len(test_dataset))

Training Samples: 120000
Testing Samples: 7600


In [17]:
from sklearn.model_selection import train_test_split

train_df_split, validation_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

print("Training Samples:", len(train_df_split))
print("Validation Samples:", len(validation_df))

Training Samples: 108000
Validation Samples: 12000


In [18]:
MAX_LENGTH = 128

train_dataset = AGNewsDataset(
    train_df_split,
    tokenizer,
    MAX_LENGTH
)

validation_dataset = AGNewsDataset(
    validation_df,
    tokenizer,
    MAX_LENGTH
)

test_dataset = AGNewsDataset(
    test_df,
    tokenizer,
    MAX_LENGTH
)

print("Training Samples:", len(train_dataset))
print("Validation Samples:", len(validation_dataset))
print("Testing Samples:", len(test_dataset))

Training Samples: 108000
Validation Samples: 12000
Testing Samples: 7600


In [19]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

print("Loading DistilBERT Model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)

print("Model loaded successfully!")

Loading DistilBERT Model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [20]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is NOT available.")
    print("Using CPU.")

GPU is NOT available.
Using CPU.
